VECTORIZATION AND BASE MODEL

In [1]:
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

df = pd.read_csv("../data/resumes_processed.csv")
X, y = df["processed"], df["Category"]

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(),
    "Naive Bayes": MultinomialNB(),
}

# 3 folds because the smallest class (PMO) has only 3 resumes
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for name, model in models.items():
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", model),
    ])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")

Logistic Regression: 0.446 (+/- 0.040)
Linear SVM: 0.892 (+/- 0.029)
Naive Bayes: 0.344 (+/- 0.070)


In [2]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LinearSVC()),
])
pred = cross_val_predict(pipe, X, y, cv=cv)
print(classification_report(y, pred, zero_division=0))

                           precision    recall  f1-score   support

                 Advocate       1.00      1.00      1.00        10
                     Arts       1.00      1.00      1.00         6
       Automation Testing       0.75      0.86      0.80         7
               Blockchain       1.00      0.80      0.89         5
         Business Analyst       0.83      0.83      0.83         6
           Civil Engineer       1.00      1.00      1.00         6
             Data Science       0.83      1.00      0.91        10
                 Database       0.83      0.91      0.87        11
          DevOps Engineer       0.83      0.71      0.77         7
         DotNet Developer       0.88      1.00      0.93         7
            ETL Developer       0.83      1.00      0.91         5
   Electrical Engineering       0.71      1.00      0.83         5
                       HR       1.00      1.00      1.00        10
                   Hadoop       1.00      1.00      1.00     